# Comparison: PhysioNet (official) vs Kaggle ("optimized/processed")

The PhysioNet PTB-XL distribution under `physionet/` is the official source. The Kaggle republish under
`kaggle/` is described by its publisher as "optimized/processed", with no changelog documenting what that
means. This notebook checks, exhaustively rather than on a single sample record, whether the two on-disk
copies actually differ at all.

Two checks, in order of cost:
1. **Fast**: compare the official `SHA256SUMS.txt` checksum manifest between the two trees directly
   (near-instant - covers every file's expected hash in one comparison).
2. **Slow, exhaustive**: independently re-hash *every* file in both trees and compare them directly against
   each other (not just against the manifest) - this is the one that actually proves the on-disk bytes
   match, regardless of whether either copy's manifest could be trusted on its own. Takes a while (the full
   PTB-XL records100+records500 trees are ~87,000 files, ~3.2 GB) - expect roughly 20-30 minutes.


In [ ]:
from pathlib import Path


def find_project_root(start=None, marker="CLAUDE.md"):
    """Walk upward from `start` (default: this notebook's cwd) until a
    directory containing `marker` is found."""
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not locate project root (missing {marker}) from {current}")


PROJECT_ROOT = find_project_root()

KAGGLE_DIR = PROJECT_ROOT / 'kaggle' / 'archive' / 'ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3'
PHYSIONET_DIR = PROJECT_ROOT / 'physionet' / 'ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3'

print('Kaggle dir exists:', KAGGLE_DIR.exists())
print('PhysioNet dir exists:', PHYSIONET_DIR.exists())


## 1. Fast check: compare the SHA256SUMS.txt manifests directly

Both distributions ship an official `SHA256SUMS.txt` listing the expected hash of every file. If these two
manifest files are themselves byte-identical, that already covers all ~87,000 files at the manifest level
in one comparison - though it doesn't yet prove either side's actual files match their own manifest (that's
what the exhaustive check below is for).


In [ ]:
import hashlib


def sha256_file(path, chunk_size=1 << 20):
    """Compute the SHA256 hex digest of a file, reading it in chunks so
    large files don't need to fit in memory at once.

    Inputs:
        path: path to the file to hash.
        chunk_size: read buffer size in bytes.

    Output:
        Hex-encoded SHA256 digest (str).
    """
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk_size), b''):
            h.update(block)
    return h.hexdigest()


kaggle_manifest_hash = sha256_file(KAGGLE_DIR / 'SHA256SUMS.txt')
physionet_manifest_hash = sha256_file(PHYSIONET_DIR / 'SHA256SUMS.txt')

print('Kaggle SHA256SUMS.txt hash   :', kaggle_manifest_hash)
print('PhysioNet SHA256SUMS.txt hash:', physionet_manifest_hash)
print('Manifests identical:', kaggle_manifest_hash == physionet_manifest_hash)


## 2. Slow, exhaustive check: hash and compare every file directly

Walks both trees, matches files by relative path, and compares them by size then by SHA256 hash - directly
against each other, not just against the manifest. This is what actually proves the Kaggle copy's bytes
match the official PhysioNet copy's bytes. **Takes ~20-30 minutes** (independently confirmed: a prior run
outside this notebook completed in 1754s / ~29 min over 87,204 files, ~3.2 GB, with zero differences found).
Skip re-running this cell if you already trust that result and just want the fast manifest check above.


In [ ]:
import os
import time


def list_relative_files(base_dir):
    """Recursively list every file under base_dir, keyed by its path
    relative to base_dir (with forward slashes, so paths compare correctly
    across OSes).

    Inputs:
        base_dir: root directory to walk.

    Output:
        dict mapping relative_path (str) -> absolute Path.
    """
    base_dir = Path(base_dir)
    result = {}
    for root, _dirs, files in os.walk(base_dir):
        for fn in files:
            abs_path = Path(root) / fn
            rel_path = abs_path.relative_to(base_dir)
            result[str(rel_path).replace(chr(92), '/')] = abs_path
    return result


def compare_datasets(dir_a, dir_b, label_a='A', label_b='B'):
    """Compare two directory trees file-by-file: which files exist only on
    one side, and among files present on both sides, which are identical vs.
    different (by size first, then by SHA256 hash).

    Inputs:
        dir_a: first directory tree to compare.
        dir_b: second directory tree to compare.
        label_a: display label for dir_a.
        label_b: display label for dir_b.

    Output:
        dict with "n_files_a"/"n_files_b" (file counts), "only_in_a"/
        "only_in_b" (relative paths present on only one side), "n_common",
        "n_identical", "different" (relative paths that differ), and
        "size_mismatch" (subset of "different" that already differs in
        size, before any hashing).
    """
    files_a = list_relative_files(dir_a)
    files_b = list_relative_files(dir_b)

    only_in_a = sorted(set(files_a) - set(files_b))
    only_in_b = sorted(set(files_b) - set(files_a))
    common = sorted(set(files_a) & set(files_b))

    identical, different, size_mismatch = [], [], []
    for rel in common:
        pa, pb = files_a[rel], files_b[rel]
        size_a, size_b = pa.stat().st_size, pb.stat().st_size
        if size_a != size_b:
            size_mismatch.append((rel, size_a, size_b))
            different.append(rel)
            continue
        if sha256_file(pa) == sha256_file(pb):
            identical.append(rel)
        else:
            different.append(rel)

    return {
        'label_a': label_a,
        'label_b': label_b,
        'n_files_a': len(files_a),
        'n_files_b': len(files_b),
        'only_in_a': only_in_a,
        'only_in_b': only_in_b,
        'n_common': len(common),
        'n_identical': len(identical),
        'different': different,
        'size_mismatch': size_mismatch,
    }


t0 = time.time()
result = compare_datasets(KAGGLE_DIR, PHYSIONET_DIR, label_a='kaggle', label_b='physionet')
elapsed = time.time() - t0

print('Files in kaggle:   ', result['n_files_a'])
print('Files in physionet:', result['n_files_b'])
print('Only in kaggle:    ', result['only_in_a'][:10])
print('Only in physionet: ', result['only_in_b'][:10])
print('Common files:      ', result['n_common'])
print('Identical:         ', result['n_identical'])
print('Different:         ', len(result['different']))
if result['different']:
    print('  first differing files:', result['different'][:20])
    print('  size mismatches:', result['size_mismatch'][:10])
print(f'Elapsed: {elapsed:.1f}s')


## Conclusion

If both checks above report zero differences (identical manifests, zero "different" files, no size
mismatches, and `n_identical == n_common == n_files_a == n_files_b`), the Kaggle copy is a byte-for-byte
match of the official PhysioNet distribution across every file in `records100/`, `records500/`, and the
top-level metadata files. "Optimized/processed" as advertised by the Kaggle listing does not correspond to
any detectable file-level change - most likely descriptive/marketing language on the listing page rather
than an actual data transformation.
